# Complete NLP Data Loader Implementation

This notebook demonstrates the full process of creating an NLP data loader in PyTorch, including:
- Custom Dataset creation
- Tokenization
- Vocabulary building
- Custom collate functions
- Batch processing with padding

## Step 1: Import Libraries and Define Sample Data

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torch.nn.utils.rnn import pad_sequence
from typing import List, Iterable

# Sample sentences for demonstration
sentences = [
    "If you want to know what a man's like, take a good look at how he treats his inferiors, not his equals.",
    "Fame's a fickle friend, Harry.",
    "It is our choices, Harry, that show what we truly are, far more than our abilities.",
    "Soon we must all face the choice between what is right and what is easy.",
    "Youth can not know how age thinks and feels. But old men are guilty if they forget what it was to be young.",
    "You are awesome!",
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is transforming the world of artificial intelligence."
]

print(f"Total sentences: {len(sentences)}")

Total sentences: 8


## Step 2: Create Custom Dataset Class

In [2]:
class CustomDataset(Dataset):
    """
    Custom PyTorch Dataset for text data.
    
    Can work in two modes:
    1. Simple mode: Just stores raw sentences
    2. Advanced mode: Includes tokenizer and vocab for preprocessing
    """
    
    def __init__(self, sentences, tokenizer=None, vocab=None):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.vocab = vocab
    
    def __len__(self):
        return len(self.sentences)
    
    def __getitem__(self, idx):
        if self.tokenizer is not None and self.vocab is not None:
            # Tokenize and convert to indices
            tokens = self.tokenizer(self.sentences[idx])
            tensor_indices = [self.vocab[token] for token in tokens]
            return torch.tensor(tensor_indices, dtype=torch.long)
        else:
            # Return raw sentence
            return self.sentences[idx]

# Test the dataset
simple_dataset = CustomDataset(sentences)
print(f"Dataset size: {len(simple_dataset)}")
print(f"Sample item: {simple_dataset[0]}")

Dataset size: 8
Sample item: If you want to know what a man's like, take a good look at how he treats his inferiors, not his equals.


## Step 3: Setup Tokenizer

In [3]:
# Create tokenizer
tokenizer = get_tokenizer("basic_english")

# Test tokenization
sample_sentence = sentences[0]
tokens = tokenizer(sample_sentence)

print(f"Original sentence: {sample_sentence}")
print(f"Tokenized: {tokens}")
print(f"Number of tokens: {len(tokens)}")

Original sentence: If you want to know what a man's like, take a good look at how he treats his inferiors, not his equals.
Tokenized: ['if', 'you', 'want', 'to', 'know', 'what', 'a', 'man', "'", 's', 'like', ',', 'take', 'a', 'good', 'look', 'at', 'how', 'he', 'treats', 'his', 'inferiors', ',', 'not', 'his', 'equals', '.']
Number of tokens: 27


## Step 4: Build Vocabulary

In [4]:
# Build vocabulary from all tokenized sentences
vocab = build_vocab_from_iterator(map(tokenizer, sentences))

# Set default index for unknown tokens
vocab.set_default_index(0)

print(f"Vocabulary size: {len(vocab)}")
print(f"\nFirst 20 tokens in vocabulary:")
for i in range(min(20, len(vocab))):
    print(f"  {i}: {vocab.get_itos()[i]}")

# Test vocabulary lookup
test_word = "the"
print(f"\nWord '{test_word}' has index: {vocab[test_word]}")
print(f"Index {vocab[test_word]} maps to: {vocab.get_itos()[vocab[test_word]]}")

Vocabulary size: 81

First 20 tokens in vocabulary:
  0: .
  1: ,
  2: what
  3: is
  4: the
  5: a
  6: are
  7: '
  8: and
  9: harry
  10: his
  11: how
  12: if
  13: it
  14: know
  15: not
  16: our
  17: s
  18: to
  19: we

Word 'the' has index: 4
Index 4 maps to: the


## Step 5: Define Custom Collate Function

In [5]:
def collate_fn(batch):
    """
    Custom collate function to process batches of variable-length sequences.
    
    Steps:
    1. Tokenize each sentence in the batch
    2. Convert tokens to vocabulary indices
    3. Pad sequences to equal length
    4. Return batched tensor
    """
    tensor_batch = []
    
    for sample in batch:
        # Tokenize the sentence
        tokens = tokenizer(sample)
        
        # Convert tokens to vocabulary indices
        indices = [vocab[token] for token in tokens]
        
        # Create tensor from indices
        tensor_batch.append(torch.tensor(indices, dtype=torch.long))
    
    # Pad sequences to have equal lengths
    # batch_first=True means output shape is [batch_size, seq_len]
    # padding_value=0 fills shorter sequences with 0
    padded_batch = pad_sequence(
        tensor_batch, 
        batch_first=True, 
        padding_value=0
    )
    
    return padded_batch

# Test collate function with a small batch
test_batch = sentences[:3]
padded_result = collate_fn(test_batch)

print(f"Input batch (3 sentences):")
for i, sent in enumerate(test_batch):
    print(f"  {i+1}. {sent[:50]}...")

print(f"\nPadded tensor shape: {padded_result.shape}")
print(f"Batch size: {padded_result.shape[0]}")
print(f"Max sequence length: {padded_result.shape[1]}")
print(f"\nPadded tensor:\n{padded_result}")

Input batch (3 sentences):
  1. If you want to know what a man's like, take a good...
  2. Fame's a fickle friend, Harry....
  3. It is our choices, Harry, that show what we truly ...

Padded tensor shape: torch.Size([3, 27])
Batch size: 3
Max sequence length: 27

Padded tensor:
tensor([[12, 20, 76, 18, 14,  2,  5, 57,  7, 17, 54,  1, 68,  5, 46, 55, 26, 11,
         48, 74, 10, 49,  1, 15, 10, 37,  0],
        [39,  7, 17,  5, 42, 45,  1,  9,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0],
        [13,  3, 16, 34,  1,  9,  1, 70, 66,  2, 19, 75,  6,  1, 40, 59, 69, 16,
         22,  0,  0,  0,  0,  0,  0,  0,  0]])


## Step 6: Create DataLoader

In [6]:
# Create dataset instance
dataset = CustomDataset(sentences)

# Define batch size
BATCH_SIZE = 3

# Create DataLoader
dataloader = DataLoader(
    dataset=dataset,           # Dataset containing raw sentences
    batch_size=BATCH_SIZE,      # Number of samples per batch
    shuffle=True,               # Shuffle data each epoch
    collate_fn=collate_fn       # Custom collate function
)

print(f"DataLoader created:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Number of batches: {len(dataloader)}")
print(f"  Total samples: {len(dataset)}")

DataLoader created:
  Batch size: 3
  Number of batches: 3
  Total samples: 8


## Step 7: Process Data in Batches

In [7]:
# Utility function to convert indices back to text
def indices_to_text(indices, vocab, remove_padding=True):
    """Convert tensor indices back to text."""
    tokens = []
    for idx in indices:
        idx_val = idx.item() if isinstance(idx, torch.Tensor) else idx
        if remove_padding and idx_val == 0:
            continue
        if idx_val < len(vocab.get_itos()):
            tokens.append(vocab.get_itos()[idx_val])
    return " ".join(tokens)

# Iterate through batches
print("Processing batches:\n")
print("=" * 70)

for batch_idx, batch in enumerate(dataloader):
    print(f"\nBatch {batch_idx + 1}:")
    print(f"  Shape: {batch.shape}  [batch_size={batch.shape[0]}, seq_len={batch.shape[1]}]")
    
    # Convert back to text for visualization
    print("  Sentences in batch:")
    for i, sequence in enumerate(batch):
        sentence = indices_to_text(sequence, vocab)
        print(f"    {i+1}. {sentence}")
    
    # Show padding
    print(f"  Padding tokens (0s) per sequence:")
    for i, sequence in enumerate(batch):
        padding_count = (sequence == 0).sum().item()
        print(f"    {i+1}. {padding_count} padding tokens")

Processing batches:


Batch 1:
  Shape: torch.Size([3, 27])  [batch_size=3, seq_len=27]
  Sentences in batch:
    1. you are awesome !
    2. if you want to know what a man ' s like , take a good look at how he treats his inferiors , not his equals
    3. soon we must all face the choice between what is right and what is easy
  Padding tokens (0s) per sequence:
    1. 23 padding tokens
    2. 1 padding tokens
    3. 12 padding tokens

Batch 2:
  Shape: torch.Size([3, 25])  [batch_size=3, seq_len=25]
  Sentences in batch:
    1. fame ' s a fickle friend , harry
    2. youth can not know how age thinks and feels but old men are guilty if they forget what it was to be young
    3. machine learning is transforming the world of artificial intelligence
  Padding tokens (0s) per sequence:
    1. 17 padding tokens
    2. 2 padding tokens
    3. 16 padding tokens

Batch 3:
  Shape: torch.Size([2, 20])  [batch_size=2, seq_len=20]
  Sentences in batch:
    1. it is our choices , harry , that show

## Step 8: Advanced - Sorting by Length to Minimize Padding

In [8]:
# Sort sentences by length to minimize padding
sorted_sentences = sorted(sentences, key=lambda x: len(tokenizer(x)))

print("Original order (first 5):")
for i, sent in enumerate(sentences[:5]):
    print(f"  {i+1}. Length {len(tokenizer(sent)):2d}: {sent[:50]}...")

print("\nSorted by length (first 5):")
for i, sent in enumerate(sorted_sentences[:5]):
    print(f"  {i+1}. Length {len(tokenizer(sent)):2d}: {sent[:50]}...")

# Create sorted dataset and dataloader
sorted_dataset = CustomDataset(sorted_sentences)
sorted_dataloader = DataLoader(
    dataset=sorted_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Don't shuffle to keep sorted order
    collate_fn=collate_fn
)

print("\n" + "=" * 70)
print("Comparing padding with sorted vs unsorted batches:")
print("=" * 70)

# Compare first batch from unsorted
unsorted_batch = next(iter(dataloader))
unsorted_padding = (unsorted_batch == 0).sum().item()

# Compare first batch from sorted
sorted_batch = next(iter(sorted_dataloader))
sorted_padding = (sorted_batch == 0).sum().item()

print(f"\nUnsorted batch padding tokens: {unsorted_padding}")
print(f"Sorted batch padding tokens: {sorted_padding}")
print(f"\nReduction: {unsorted_padding - sorted_padding} fewer padding tokens!")

Original order (first 5):
  1. Length 27: If you want to know what a man's like, take a good...
  2. Length  9: Fame's a fickle friend, Harry....
  3. Length 20: It is our choices, Harry, that show what we truly ...
  4. Length 16: Soon we must all face the choice between what is r...
  5. Length 25: Youth can not know how age thinks and feels. But o...

Sorted by length (first 5):
  1. Length  4: You are awesome!...
  2. Length  9: Fame's a fickle friend, Harry....
  3. Length 10: The quick brown fox jumps over the lazy dog....
  4. Length 10: Machine learning is transforming the world of arti...
  5. Length 16: Soon we must all face the choice between what is r...

Comparing padding with sorted vs unsorted batches:

Unsorted batch padding tokens: 18
Sorted batch padding tokens: 9

Reduction: 9 fewer padding tokens!


## Step 9: Understanding batch_first Parameter

In [9]:
# Compare batch_first=True vs False
test_sequences = [
    torch.tensor([1, 2, 3, 4]),
    torch.tensor([5, 6, 7]),
    torch.tensor([8, 9, 10, 11, 12])
]

# With batch_first=True (default for most models)
padded_true = pad_sequence(test_sequences, batch_first=True, padding_value=0)
print("With batch_first=True:")
print(f"  Shape: {padded_true.shape}  [batch_size, seq_len]")
print(f"  Tensor:\n{padded_true}")

print("\n" + "-" * 50 + "\n")

# With batch_first=False (for RNNs)
padded_false = pad_sequence(test_sequences, batch_first=False, padding_value=0)
print("With batch_first=False:")
print(f"  Shape: {padded_false.shape}  [seq_len, batch_size]")
print(f"  Tensor:\n{padded_false}")

print("\n" + "=" * 70)
print("Note: Most modern models (Transformers) use batch_first=True")
print("RNNs typically use batch_first=False")

With batch_first=True:
  Shape: torch.Size([3, 5])  [batch_size, seq_len]
  Tensor:
tensor([[ 1,  2,  3,  4,  0],
        [ 5,  6,  7,  0,  0],
        [ 8,  9, 10, 11, 12]])

--------------------------------------------------

With batch_first=False:
  Shape: torch.Size([5, 3])  [seq_len, batch_size]
  Tensor:
tensor([[ 1,  5,  8],
        [ 2,  6,  9],
        [ 3,  7, 10],
        [ 4,  0, 11],
        [ 0,  0, 12]])

Note: Most modern models (Transformers) use batch_first=True
RNNs typically use batch_first=False


## Step 10: Complete Workflow Summary

In [10]:
print("=" * 70)
print("COMPLETE NLP DATA LOADER WORKFLOW")
print("=" * 70)

print("""
1. ✅ Define your text data (sentences/corpus)
2. ✅ Create CustomDataset class (inherits from torch.utils.data.Dataset)
   - Implement __init__, __len__, and __getitem__ methods
3. ✅ Setup tokenizer (get_tokenizer from torchtext)
   - Options: 'basic_english', 'spacy', etc.
4. ✅ Build vocabulary (build_vocab_from_iterator)
   - Maps tokens to numerical indices
   - Set default index for unknown tokens
5. ✅ Create dataset instance
   - Can store raw text or pre-processed tensors
6. ✅ Define custom collate function
   - Tokenize → Numericalize → Pad sequences
   - Handles variable-length sequences
7. ✅ Create DataLoader
   - Specify batch_size, shuffle, collate_fn
8. ✅ Iterate through batches for training

Key Concepts:
- Datasets: Store and retrieve individual samples
- DataLoaders: Batch, shuffle, and iterate over data
- Collate Functions: Handle variable-length sequences via padding
- Padding: Makes sequences equal length for batch processing
- Sorting: Minimizes padding overhead
- batch_first: Controls tensor shape [batch, seq] vs [seq, batch]

This pipeline is ready for training NLP models!
""")

COMPLETE NLP DATA LOADER WORKFLOW

1. ✅ Define your text data (sentences/corpus)
2. ✅ Create CustomDataset class (inherits from torch.utils.data.Dataset)
   - Implement __init__, __len__, and __getitem__ methods
3. ✅ Setup tokenizer (get_tokenizer from torchtext)
   - Options: 'basic_english', 'spacy', etc.
4. ✅ Build vocabulary (build_vocab_from_iterator)
   - Maps tokens to numerical indices
   - Set default index for unknown tokens
5. ✅ Create dataset instance
   - Can store raw text or pre-processed tensors
6. ✅ Define custom collate function
   - Tokenize → Numericalize → Pad sequences
   - Handles variable-length sequences
7. ✅ Create DataLoader
   - Specify batch_size, shuffle, collate_fn
8. ✅ Iterate through batches for training

Key Concepts:
- Datasets: Store and retrieve individual samples
- DataLoaders: Batch, shuffle, and iterate over data
- Collate Functions: Handle variable-length sequences via padding
- Padding: Makes sequences equal length for batch processing
- Sortin

## Bonus: Real-World Example - Translation Dataset

In [11]:
# Example for translation task (source and target pairs)
translation_pairs = [
    ("Hello world", "Hola mundo"),
    ("How are you?", "¿Cómo estás?"),
    ("I love machine learning", "Me encanta el aprendizaje automático"),
    ("Good morning", "Buenos días")
]

class TranslationDataset(Dataset):
    def __init__(self, pairs, src_tokenizer, tgt_tokenizer, src_vocab, tgt_vocab):
        self.pairs = pairs
        self.src_tokenizer = src_tokenizer
        self.tgt_tokenizer = tgt_tokenizer
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]
        
        src_tokens = self.src_tokenizer(src_text)
        src_indices = [self.src_vocab[token] for token in src_tokens]
        
        tgt_tokens = self.tgt_tokenizer(tgt_text)
        tgt_indices = [self.tgt_vocab[token] for token in tgt_tokens]
        
        return torch.tensor(src_indices), torch.tensor(tgt_indices)

def translation_collate_fn(batch):
    """Collate function for translation pairs."""
    src_batch, tgt_batch = [], []
    
    for src_seq, tgt_seq in batch:
        src_batch.append(src_seq)
        tgt_batch.append(tgt_seq)
    
    # Pad both source and target sequences
    src_padded = pad_sequence(src_batch, batch_first=True, padding_value=0)
    tgt_padded = pad_sequence(tgt_batch, batch_first=True, padding_value=0)
    
    return src_padded, tgt_padded

# Build separate vocabularies for source and target
src_tokenizer = get_tokenizer("basic_english")
tgt_tokenizer = get_tokenizer("basic_english")

src_texts = [pair[0] for pair in translation_pairs]
tgt_texts = [pair[1] for pair in translation_pairs]

src_vocab = build_vocab_from_iterator(map(src_tokenizer, src_texts))
tgt_vocab = build_vocab_from_iterator(map(tgt_tokenizer, tgt_texts))

# Create translation dataset
trans_dataset = TranslationDataset(
    translation_pairs, src_tokenizer, tgt_tokenizer, src_vocab, tgt_vocab
)

trans_dataloader = DataLoader(
    trans_dataset, batch_size=2, shuffle=True, collate_fn=translation_collate_fn
)

print("Translation Dataset Example:")
for src_batch, tgt_batch in trans_dataloader:
    print(f"\nSource batch shape: {src_batch.shape}")
    print(f"Target batch shape: {tgt_batch.shape}")
    break

Translation Dataset Example:

Source batch shape: torch.Size([2, 2])
Target batch shape: torch.Size([2, 2])
